# Qubit-Effcient Graph-Decomposed Quantum Approximate Optimization for Hydrogen-Integrated Multi-Energy Systems: A Constraint-Preserving Hybrid Quantum–Classical Framework with Verified Energy Scheduling and Computational Resource Analysis
**This single notebook regenerates EVERYTHING from scratch:**
- `results_authoritative.json` — complete results (14 sections) + provenance (checksums, seed, versions)
- `figures/*.png` — all data & experimental figures (300 DPI)
- `tables/*.csv` — all result tables in CSV form
- `hagdcp_outputs.zip` — all of the above bundled for one-click download

### How to run on Kaggle (3 steps)
1. **Create** a Kaggle Notebook → *File → Import Notebook* → upload *only this `.ipynb`*.
2. **Add Input** (right panel) → add the **OPSD "Time series" 2020-10-06** dataset (`time_series_60min_singleindex.csv`). *If not found, upload your `Dataset.zip` as a Kaggle Dataset and add it — this notebook auto-extracts `.zip`.*
3. **Run All.** (~4 min). Download `hagdcp_outputs.zip` from the right panel. Numbers match the manuscript (seed = 2026).

No other file is required — the full corrected code is embedded below.

In [1]:
# Cell 1 — install only missing dependencies
import importlib.util, sys, subprocess
need={'highspy':'highspy==1.11.0','qiskit':'qiskit==1.4.6','qiskit_aer':'qiskit-aer==0.17.2',
      'networkx':'networkx','scipy':'scipy','pandas':'pandas','numpy':'numpy','matplotlib':'matplotlib'}
missing=[v for m,v in need.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing]); print('installed:',missing)
else: print('all dependencies present')

installed: ['qiskit==1.4.6', 'qiskit-aer==0.17.2']


In [3]:
# Cell 2 — corrected HA-GDCP-QAOA implementation (self-contained; fixes H1-H14)
"""
HA-GDCP-QAOA — corrected, self-contained implementation.
Addresses code-audit findings H1-H14. No machine-specific paths; auto-discovers OPSD.
"""
import os, glob, zipfile, hashlib, itertools, time, json
import numpy as np, pandas as pd, networkx as nx, highspy
from scipy.optimize import minimize
from scipy.stats import spearmanr, kendalltau, wilcoxon

# ============================ CONFIG (H16: configurable paths) ============================
def find_opsd():
    env=os.environ.get('OPSD_CSV')
    if env and os.path.exists(env): return env
    pats=['/kaggle/input/**/time_series_60min_singleindex.csv',
          './**/time_series_60min_singleindex.csv',
          os.path.expanduser('~/**/time_series_60min_singleindex.csv')]
    for p in pats:
        hits=glob.glob(p,recursive=True)
        if hits: return hits[0]
    zips=glob.glob('/kaggle/input/**/*.zip',recursive=True)+glob.glob('./*.zip')
    for z in zips:
        with zipfile.ZipFile(z) as f: f.extractall('opsd_data')
    hits=glob.glob('./**/time_series_60min_singleindex.csv',recursive=True)
    if hits: return hits[0]
    raise FileNotFoundError('OPSD time_series_60min_singleindex.csv not found')

CONFIG=dict(seed=2026, block=6, qaoa_p=3, restarts=4, maxiter=200, shots=256,
            allow_simultaneous_el_fc=True,   # H6: documented policy (never optimal; verified)
            facility=dict(load_peak_kw=400.0, pv_kw=200.0, wind_kw=150.0))

PARAMS=dict(Pel_max=100.0,Pel_min=20.0,Pfc_max=60.0,Pfc_min=12.0,Php_max=90.0,
    eta_el=0.65,eta_fc=0.50,e_H=33.33,R_el=60.0,Hmin=0.0,Hmax=50.0,H0=10.0,Hterm=10.0,
    self_discharge=0.0,   # H8: fractional standing loss, APPLIED in balance (default 0)
    cdeg_el=0.004,cdeg_fc=0.006,cstart=0.05,lamH=5.0,lamQ=0.3,csell=0.9,dt=1.0,Pgrid_max=500.0)

# ============================ data / benchmark ============================
def build_year(csv, params=PARAMS, seed=2026):
    cols=['utc_timestamp','DE_load_actual_entsoe_transparency','DE_solar_generation_actual',
          'DE_wind_generation_actual','DE_LU_price_day_ahead']
    df=pd.read_csv(csv,usecols=cols,parse_dates=['utc_timestamp']).set_index('utc_timestamp').sort_index()
    m=(df.index>=pd.Timestamp('2019-01-01',tz='UTC'))&(df.index<pd.Timestamp('2020-01-01',tz='UTC'))
    d=df.loc[m].copy(); d['DE_LU_price_day_ahead']=d['DE_LU_price_day_ahead'].ffill().bfill()
    out=pd.DataFrame(index=d.index)
    out['load_kw']=d['DE_load_actual_entsoe_transparency']/d['DE_load_actual_entsoe_transparency'].max()*params['Pel_max']*0+ \
                   d['DE_load_actual_entsoe_transparency']/d['DE_load_actual_entsoe_transparency'].max()*400.0
    out['solar_kw']=d['DE_solar_generation_actual']/30028.0*200.0
    out['wind_kw']=d['DE_wind_generation_actual']/45085.0*150.0
    out['price_eur_kwh']=d['DE_LU_price_day_ahead']/1000.0
    doy=out.index.dayofyear.values; hod=out.index.hour.values
    seasonal=0.5+0.5*np.cos(2*np.pi*(doy-15)/365.0); diurnal=0.6+0.4*np.sin(2*np.pi*(hod-7)/24.0).clip(min=-1)
    out['heat_kwth']=(20.0+70.0*seasonal)*(0.7+0.3*diurnal)
    amb=5.0+15.0*(1-seasonal)-4.0*np.cos(2*np.pi*(hod-14)/24.0); out['cop']=(2.2+0.08*amb).clip(1.8,4.5)
    rng=np.random.default_rng(seed); base=0.45+0.25*seasonal; daily=0.8+0.4*((hod>=6)&(hod<=20))
    out['h2_demand_kg_h']=(base*daily*rng.normal(1.0,0.05,len(out)).clip(0.85,1.15)).round(3)
    return out
def split_chrono(df): return df.loc[:'2019-08-31'],df.loc['2019-09-01':'2019-10-31'],df.loc['2019-11-01':]
def make_instance(df,start,T):
    w=df.loc[start:].iloc[:T]
    return dict(key=f'{start}|{T}',ts=w.index,load=w['load_kw'].values,solar=w['solar_kw'].values,
        wind=w['wind_kw'].values,price=w['price_eur_kwh'].values,heat=w['heat_kwth'].values,
        cop=w['cop'].values,h2d=w['h2_demand_kg_h'].values,T=T)

# ============================ H2: safe cache key (full hash) ============================
_LP_CACHE={}
def _inst_hash(inst,p):
    h=hashlib.sha256()
    for k in ['load','solar','wind','price','heat','cop','h2d']:
        h.update(np.ascontiguousarray(inst[k],dtype=np.float64).tobytes())
    for k in sorted(p): h.update(f'{k}={p[k]}'.encode())
    h.update(str(inst['T']).encode())
    return h.hexdigest()

# ============================ H1,H6,H7,H8: continuous recourse LP ============================
INF=highspy.kHighsInf
def lp_recourse(inst,uel,ufc,p=PARAMS,return_sched=False,use_cache=True):
    T=inst['T']
    if use_cache and not return_sched:
        ck=(_inst_hash(inst,p),tuple(int(x) for x in uel),tuple(int(x) for x in ufc))
        if ck in _LP_CACHE: return _LP_CACHE[ck]
    load,sol,wnd,price=inst['load'],inst['solar'],inst['wind'],inst['price']
    heat,cop,h2d=inst['heat'],inst['cop'],inst['h2d']
    h=highspy.Highs(); h.setOptionValue('output_flag',False)
    nv=9; idx=lambda t,j:t*nv+j; base=T*nv; Hi=lambda t:base+t; N=base+(T+1)
    lo=np.zeros(N); up=np.full(N,INF); cost=np.zeros(N)
    for t in range(T):
        up[idx(t,0)]=p['Pgrid_max']; up[idx(t,1)]=p['Pgrid_max']            # H1 grid cap
        up[idx(t,2)]=max(sol[t],0); up[idx(t,3)]=max(wnd[t],0)
        up[idx(t,4)]=p['Pel_max']*uel[t]; lo[idx(t,4)]=p['Pel_min']*uel[t]
        up[idx(t,5)]=p['Pfc_max']*ufc[t]; lo[idx(t,5)]=p['Pfc_min']*ufc[t]
        up[idx(t,6)]=p['Php_max']; up[idx(t,7)]=h2d[t]
        cost[idx(t,0)]=price[t]*p['dt']; cost[idx(t,1)]=-p['csell']*max(price[t],0)*p['dt']
        cost[idx(t,4)]=p['cdeg_el']; cost[idx(t,5)]=p['cdeg_fc']; cost[idx(t,7)]=p['lamH']; cost[idx(t,8)]=p['lamQ']
    for t in range(T+1): lo[Hi(t)]=p['Hmin']; up[Hi(t)]=p['Hmax']
    lo[Hi(0)]=up[Hi(0)]=p['H0']; lo[Hi(T)]=max(p['Hterm'],p['Hmin'])
    h.addVars(N,lo,up); h.changeColsCost(N,np.arange(N),cost)
    starts=[];ind=[];val=[];rl=[];ru=[]
    def row(pairs,l,u):
        starts.append(len(ind))
        for c,v in pairs: ind.append(int(c)); val.append(float(v))
        rl.append(l); ru.append(u)
    sd=p['self_discharge']
    for t in range(T):
        row([(idx(t,2),1),(idx(t,3),1),(idx(t,0),1),(idx(t,5),1),(idx(t,4),-1),(idx(t,6),-1),(idx(t,1),-1)],load[t],load[t])
        row([(idx(t,6),cop[t]),(idx(t,8),1)],heat[t],heat[t])
        # H8: H[t+1] = (1-sd)H[t] + dt(eta_el Pel/eH - Pfc/(eta_fc eH) - d + qH)
        row([(Hi(t+1),1),(Hi(t),-(1-sd)),(idx(t,4),-p['dt']*p['eta_el']/p['e_H']),
             (idx(t,5),p['dt']/(p['eta_fc']*p['e_H'])),(idx(t,7),-p['dt'])],-p['dt']*h2d[t],-p['dt']*h2d[t])
        row([(idx(t,1),1),(idx(t,2),-1),(idx(t,3),-1),(idx(t,5),-1)],-INF,0.0)   # H1 export<=generation
        if t>0: row([(idx(t,4),1),(idx(t-1,4),-1)],-p['R_el'],p['R_el'])
    h.addRows(len(rl),np.array(rl),np.array(ru),len(ind),np.array(starts),np.array(ind,dtype=np.int32),np.array(val))
    h.run()
    st=h.modelStatusToString(h.getModelStatus())                               # H7 status check
    if st!='Optimal':
        res=(None,None) if return_sched else None
        if use_cache and not return_sched: _LP_CACHE[ck]=None
        return res
    start_cost=sum(p['cstart']*max(0,uel[t]-(uel[t-1] if t>0 else 0)) for t in range(T))
    J=h.getObjectiveValue()+start_cost
    if return_sched: return J,np.array(h.getSolution().col_value)
    if use_cache: _LP_CACHE[ck]=J
    return J

# ============================ exact MILP (H6 optional mutual-excl, H7 status) ============================
def milp_highs(inst,p=PARAMS,time_limit=60.0,mutual_excl=None):
    if mutual_excl is None: mutual_excl=not CONFIG['allow_simultaneous_el_fc']
    T=inst['T']; load,sol,wnd,price=inst['load'],inst['solar'],inst['wind'],inst['price']
    heat,cop,h2d=inst['heat'],inst['cop'],inst['h2d']
    h=highspy.Highs(); h.setOptionValue('output_flag',False); h.setOptionValue('time_limit',time_limit)
    nv=9; idx=lambda t,j:t*nv+j; baseH=T*nv; Hi=lambda t:baseH+t
    baseB=baseH+(T+1); Uel=lambda t:baseB+t; Ufc=lambda t:baseB+T+t; Vel=lambda t:baseB+2*T+t; N=baseB+3*T
    lo=np.zeros(N); up=np.full(N,INF); cost=np.zeros(N); integ=np.zeros(N,dtype=np.int32)
    for t in range(T):
        up[idx(t,0)]=p['Pgrid_max']; up[idx(t,1)]=p['Pgrid_max']
        up[idx(t,2)]=max(sol[t],0); up[idx(t,3)]=max(wnd[t],0)
        up[idx(t,4)]=p['Pel_max']; up[idx(t,5)]=p['Pfc_max']; up[idx(t,6)]=p['Php_max']; up[idx(t,7)]=h2d[t]
        cost[idx(t,0)]=price[t]*p['dt']; cost[idx(t,1)]=-p['csell']*max(price[t],0)*p['dt']
        cost[idx(t,4)]=p['cdeg_el']; cost[idx(t,5)]=p['cdeg_fc']; cost[idx(t,7)]=p['lamH']; cost[idx(t,8)]=p['lamQ']
    for t in range(T+1): up[Hi(t)]=p['Hmax']
    lo[Hi(0)]=up[Hi(0)]=p['H0']; lo[Hi(T)]=max(p['Hterm'],0.0)
    for t in range(T):
        for b in (Uel(t),Ufc(t),Vel(t)): up[b]=1.0; integ[b]=1
        cost[Vel(t)]=p['cstart']
    h.addVars(N,lo,up); h.changeColsCost(N,np.arange(N),cost); h.changeColsIntegrality(N,np.arange(N),integ)
    starts=[];ind=[];val=[];rl=[];ru=[]
    def row(pairs,l,u):
        starts.append(len(ind))
        for c,v in pairs: ind.append(int(c)); val.append(float(v))
        rl.append(l); ru.append(u)
    sd=p['self_discharge']
    for t in range(T):
        row([(idx(t,2),1),(idx(t,3),1),(idx(t,0),1),(idx(t,5),1),(idx(t,4),-1),(idx(t,6),-1),(idx(t,1),-1)],load[t],load[t])
        row([(idx(t,6),cop[t]),(idx(t,8),1)],heat[t],heat[t])
        row([(Hi(t+1),1),(Hi(t),-(1-sd)),(idx(t,4),-p['dt']*p['eta_el']/p['e_H']),
             (idx(t,5),p['dt']/(p['eta_fc']*p['e_H'])),(idx(t,7),-p['dt'])],-p['dt']*h2d[t],-p['dt']*h2d[t])
        row([(idx(t,4),1),(Uel(t),-p['Pel_max'])],-INF,0.0); row([(idx(t,4),1),(Uel(t),-p['Pel_min'])],0.0,INF)
        row([(idx(t,5),1),(Ufc(t),-p['Pfc_max'])],-INF,0.0); row([(idx(t,5),1),(Ufc(t),-p['Pfc_min'])],0.0,INF)
        if t==0: row([(Vel(t),1),(Uel(t),-1)],0.0,INF)
        else: row([(Vel(t),1),(Uel(t),-1),(Uel(t-1),1)],0.0,INF)
        if t>0: row([(idx(t,4),1),(idx(t-1,4),-1)],-p['R_el'],p['R_el'])
        row([(idx(t,1),1),(idx(t,2),-1),(idx(t,3),-1),(idx(t,5),-1)],-INF,0.0)
        if mutual_excl: row([(Uel(t),1),(Ufc(t),1)],-INF,1.0)   # H6 u_el+u_fc<=1
    h.addRows(len(rl),np.array(rl),np.array(ru),len(ind),np.array(starts),np.array(ind,dtype=np.int32),np.array(val))
    h.run(); st=h.modelStatusToString(h.getModelStatus())
    if st!='Optimal':                                           # H7: do not read garbage
        return dict(J=None,status=st,uel=None,ufc=None)
    solv=np.array(h.getSolution().col_value); info=h.getInfo()
    return dict(J=h.getObjectiveValue(),status=st,
                bound=getattr(info,'mip_dual_bound',None),gap=getattr(info,'mip_gap',None),
                uel=tuple(int(round(solv[Uel(t)])) for t in range(T)),
                ufc=tuple(int(round(solv[Ufc(t)])) for t in range(T)))

def milp_exact(inst,p=PARAMS):
    T=inst['T']
    if 2**(2*T)<=4096:
        best=None; feas=0
        for bits in itertools.product([0,1],repeat=2*T):
            uel=bits[:T]; ufc=bits[T:]
            if (not CONFIG['allow_simultaneous_el_fc']) and any(a and b for a,b in zip(uel,ufc)): continue
            c=lp_recourse(inst,uel,ufc,p)
            if c is not None:
                feas+=1
                if best is None or c<best[0]: best=(c,uel,ufc)
        return dict(J=best[0],uel=best[1],ufc=best[2],feasible_patterns=feas,total_patterns=2**(2*T),method='enum')
    return milp_highs(inst,p)

def balance_residuals(inst,sched,p=PARAMS):
    T=inst['T']; nv=9; idx=lambda t,j:t*nv+j; Hi=lambda t:T*nv+t
    load,sol,wnd=inst['load'],inst['solar'],inst['wind']; heat,cop,h2d=inst['heat'],inst['cop'],inst['h2d']; sd=p['self_discharge']
    re=rh=rH=0.0
    for t in range(T):
        e=sched[idx(t,2)]+sched[idx(t,3)]+sched[idx(t,0)]+sched[idx(t,5)]-sched[idx(t,4)]-sched[idx(t,6)]-sched[idx(t,1)]-load[t]
        hq=cop[t]*sched[idx(t,6)]+sched[idx(t,8)]-heat[t]
        Hb=sched[Hi(t+1)]-(1-sd)*sched[Hi(t)]-p['dt']*(p['eta_el']*sched[idx(t,4)]/p['e_H']-sched[idx(t,5)]/(p['eta_fc']*p['e_H'])-h2d[t]+sched[idx(t,7)])
        re=max(re,abs(e)); rh=max(rh,abs(hq)); rH=max(rH,abs(Hb))
    return dict(elec=re,heat=rh,h2=rH)

def demand_audit(inst,uel,ufc,p=PARAMS):
    T=inst['T']; _,s=lp_recourse(inst,uel,ufc,p,return_sched=True,use_cache=False)
    nv=9; idx=lambda t,j:t*nv+j
    uH=sum(s[idx(t,7)] for t in range(T)); uQ=sum(s[idx(t,8)] for t in range(T))
    simult_grid=sum(1 for t in range(T) if s[idx(t,0)]>1e-6 and s[idx(t,1)]>1e-6)
    simult_elfc=sum(1 for t in range(T) if uel[t] and ufc[t])
    return dict(pct_H2_served=round(100*(1-uH/max(inst['h2d'].sum(),1e-9)),3),
                pct_heat_served=round(100*(1-uQ/max(inst['heat'].sum(),1e-9)),3),
                simultaneous_grid_hrs=simult_grid, simultaneous_el_fc_hrs=simult_elfc)

# ============================ interaction graph ============================
def build_interaction_graph(inst):
    T=inst['T']; G=nx.Graph(); h2d=inst['h2d']; H2=max(h2d.mean(),1e-6)
    for t in range(T): G.add_node(('el',t)); G.add_node(('fc',t))
    for t in range(T):
        G.add_edge(('el',t),('fc',t),weight=1.0,kind='bus')
        if t>0: G.add_edge(('el',t),('el',t-1),weight=0.8,kind='ramp')
    for t in range(T):
        for s in range(t+1,T):
            w=np.exp(-(s-t)/max(T/3,1))*(h2d[s]/H2)
            if w>0.05: G.add_edge(('el',t),('fc',s),weight=round(0.6*w,4),kind='storage')
    ws=[d['weight'] for *_,d in G.edges(data=True)]; mx=max(ws) if ws else 1
    for *_,d in G.edges(data=True): d['weight']=round(d['weight']/mx,4)
    return G

# ============================ H4: partitioners that ALL enforce the qubit cap ============================
def _enforce_cap(parts, cap_nodes):
    """Split any block exceeding cap_nodes into chunks of <= cap_nodes (deterministic)."""
    from collections import defaultdict
    groups=defaultdict(list)
    for n,b in parts.items(): groups[b].append(n)
    newp={}; nid=0
    for b,nodes in groups.items():
        nodes=sorted(nodes,key=lambda x:(x[1],x[0]))
        for i in range(0,len(nodes),cap_nodes):
            for n in nodes[i:i+cap_nodes]: newp[n]=nid
            nid+=1
    return newp
def partition_temporal(G,T,block):  return {n:(n[1]//block) for n in G.nodes()}
def partition_carrier(G,T,block):   return {n:(0 if n[0]=='el' else 1)*((T//block)+1)+(n[1]//block) for n in G.nodes()}
def partition_random(G,T,block,seed=0):
    rng=np.random.default_rng(seed); nodes=list(G.nodes()); rng.shuffle(nodes)
    k=max(1,len(nodes)//(2*block)); return {n:i%k for i,n in enumerate(nodes)}
def partition_h2aware(G,T,block):
    target=2*block; parts={}; nid=[0]
    def rec(sub):
        if len(sub)<=target or len(sub)<=2:
            pid=nid[0]; nid[0]+=1
            for n in sub: parts[n]=pid
            return
        H=G.subgraph(sub)
        try: A,B=nx.algorithms.community.kernighan_lin_bisection(H,weight='weight',seed=1)
        except Exception: A=set(list(sub)[:len(sub)//2]); B=set(sub)-A
        rec(list(A)); rec(list(B))
    rec(list(G.nodes())); return parts
def partition_metis_like(G,T,block):
    comms=list(nx.algorithms.community.greedy_modularity_communities(G,weight='weight'))
    parts={}
    for i,c in enumerate(comms):
        for n in c: parts[n]=i
    return parts
def get_partition(name,G,T,block):
    cap=2*block
    raw={'temporal':partition_temporal,'carrier':partition_carrier,'random':partition_random,
         'h2aware':partition_h2aware,'metis-like':partition_metis_like}[name]
    parts=raw(G,T,block,0) if name=='random' else raw(G,T,block)
    return _enforce_cap(parts,cap)   # H4: hard cap for EVERY partitioner
def partition_metrics(G,parts):
    cut=tot=0.0; interface=set()
    for u,v,d in G.edges(data=True):
        tot+=d['weight']
        if parts[u]!=parts[v]: cut+=d['weight']; interface.add(u); interface.add(v)
    sizes={}
    for n,b in parts.items(): sizes[b]=sizes.get(b,0)+1
    s=list(sizes.values())
    return dict(cut_ratio=round(cut/tot,4) if tot else 0,n_blocks=len(sizes),
                max_block=max(s),min_block=min(s),interface_size=len(interface))

# ============================ QUBO surrogate (H11: held-out fidelity) ============================
def build_qubo_surrogate(inst,p=PARAMS,seed=0,holdout=0.3):
    """Fit degree-2 QUBO surrogate. Uses UNIQUE bitstrings with a disjoint train/test split
    (no leakage; verified). For n<=12 enumerates all 2^n; else samples unique rows."""
    T=inst['T']; n=2*T; rng=np.random.default_rng(seed)
    if n<=12:
        Z=np.array(list(itertools.product([0,1],repeat=n)))
    else:
        seen=set(); rows=[]
        while len(rows)<min(2**n,2000):
            z=tuple(int(x) for x in rng.integers(0,2,n))
            if z not in seen: seen.add(z); rows.append(z)
        Z=np.array(rows)
    raw=[lp_recourse(inst,tuple(z[:T]),tuple(z[T:]),p) for z in Z]
    fc=[c for c in raw if c is not None]; cmin,cmax=min(fc),max(fc); rc=max(cmax-cmin,1.0)
    inf_t=cmax+3*rc; y=np.array([c if c is not None else inf_t for c in raw])
    idx=np.arange(len(Z)); rng.shuffle(idx); cut=int((1-holdout)*len(idx)); tr,te=idx[:cut],idx[cut:]
    assert len(set(map(tuple,Z[tr]))&set(map(tuple,Z[te])))==0, 'train/test overlap!'  # zero-leakage check
    def feats(ii):
        cols=[np.ones(len(ii))]
        for i in range(n): cols.append(Z[ii,i].astype(float))
        for i in range(n):
            for j in range(i+1,n): cols.append((Z[ii,i]*Z[ii,j]).astype(float))
        return np.array(cols).T
    A=feats(tr); coef,*_=np.linalg.lstsq(A,y[tr],rcond=None)
    Q=np.zeros((n,n)); c0=coef[0]
    for i in range(n): Q[i,i]=coef[1+i]
    kk=1+n
    for i in range(n):
        for j in range(i+1,n): Q[i,j]=coef[kk]; kk+=1
    pred_te=feats(te)@coef; oos_rmse=float(np.sqrt(np.mean((pred_te-y[te])**2)))
    feas_te=np.array([raw[i] is not None for i in te])
    sp=float(spearmanr(pred_te[feas_te],y[te][feas_te]).correlation) if feas_te.sum()>2 else np.nan
    return Q,c0,n,dict(oos_rmse=oos_rmse,spearman_oos=sp,n_train=len(tr),n_test=len(te),unique=True)

def qubo_energy(Q,c0,z):
    z=np.asarray(z,float); e=c0; n=len(z)
    for i in range(n):
        e+=Q[i,i]*z[i]
        for j in range(i+1,n): e+=Q[i,j]*z[i]*z[j]
    return float(e)
def qubo_to_ising(Q,c0):
    n=Q.shape[0]; h=np.zeros(n); J=np.zeros((n,n)); const=c0
    for i in range(n):
        h[i]+=-Q[i,i]/2; const+=Q[i,i]/2
        for j in range(i+1,n): J[i,j]+=Q[i,j]/4; h[i]+=-Q[i,j]/4; h[j]+=-Q[i,j]/4; const+=Q[i,j]/4
    return h,J,const
def energy_vector(efn,n): return np.array([efn([(k>>(n-1-q))&1 for q in range(n)]) for k in range(2**n)])

# ============================ fast exact QAOA (X mixer) ============================
def fast_qaoa_probs(evec,params,p,n):
    g=params[:p]; b=params[p:]; psi=np.ones(2**n,dtype=complex)/np.sqrt(2**n); shp=[2]*n
    for l in range(p):
        psi=psi*np.exp(-1j*g[l]*evec); psi=psi.reshape(shp); cb,sb=np.cos(b[l]),np.sin(b[l])
        for q in range(n): psi=cb*psi-1j*sb*np.flip(psi,axis=q)
        psi=psi.reshape(-1)
    return np.abs(psi)**2
def optimize_qaoa_x(evec,p,n,restarts=4,seed=0,maxiter=200):
    rng=np.random.default_rng(seed); best=None; f=lambda x:float(fast_qaoa_probs(evec,x,p,n)@evec)
    for _ in range(restarts):
        r=minimize(f,rng.uniform(0,np.pi,2*p),method='COBYLA',options={'maxiter':maxiter})
        if best is None or r.fun<best.fun: best=r
    probs=fast_qaoa_probs(evec,best.x,p,n); return dict(params=best.x,probs=probs,nfev=best.nfev)
def sample_bits(probs,n,shots,rng):
    ks=rng.choice(len(probs),size=shots,p=probs/probs.sum())
    return [tuple((int(k)>>(n-1-q))&1 for q in range(n)) for k in ks]

# ============================ H14: node-based hybrid on ACTUAL partition; H5 safe failure ============================
def _node_surrogate(inst,nodes,cur_uel,cur_ufc,p,seed):
    """Fit a QUBO over the block's commitment NODES (others held at current values).
    nodes: list of ('el'|'fc', t). Returns Q,c0,k and a decoder."""
    k=len(nodes); rng=np.random.default_rng(seed)
    Z=np.array(list(itertools.product([0,1],repeat=k))) if k<=10 else rng.integers(0,2,(400,k))
    def apply(z):
        tu=list(cur_uel); tf=list(cur_ufc)
        for bit,(c,t) in zip(z,nodes):
            if c=='el': tu[t]=int(bit)
            else: tf[t]=int(bit)
        return tu,tf
    raw=[]
    for z in Z:
        tu,tf=apply(z); raw.append(lp_recourse(inst,tu,tf,p))
    fc=[c for c in raw if c is not None]
    if not fc: return None
    cmin,cmax=min(fc),max(fc); rc=max(cmax-cmin,1.0); inf_t=cmax+3*rc
    y=np.array([c if c is not None else inf_t for c in raw])
    cols=[np.ones(len(Z))]
    for i in range(k): cols.append(Z[:,i].astype(float))
    for i in range(k):
        for j in range(i+1,k): cols.append((Z[:,i]*Z[:,j]).astype(float))
    A=np.array(cols).T; coef,*_=np.linalg.lstsq(A,y,rcond=None)
    Q=np.zeros((k,k)); c0=coef[0]
    for i in range(k): Q[i,i]=coef[1+i]
    kk=1+k
    for i in range(k):
        for j in range(i+1,k): Q[i,j]=coef[kk]; kk+=1
    return Q,c0,k,apply

def hybrid_solve(inst,partition,p=PARAMS,qp=3,shots=256,seed=0):
    """Decompose on the partition's actual node-blocks (H14). Each block's qubits = its node count (<=cap).
    Returns status + (uel,ufc) or None (H5). Logs resources (H13,H15)."""
    T=inst['T']; rng=np.random.default_rng(seed)
    blocks={}
    for node,b in partition.items(): blocks.setdefault(b,[]).append(node)
    node_blocks=[sorted(v,key=lambda x:(x[1],x[0])) for v in blocks.values()]
    peak=max(len(b) for b in node_blocks)   # qubits = nodes in largest block (<= cap)
    cur_uel=[0]*T; cur_ufc=[0]*T
    res=dict(circuit_evals=0,two_qubit_gates=0,recourse_calls=0,peak_qubits=peak,
             n_blocks=len(node_blocks),total_circuit_depth=0)
    for nodes in node_blocks:
        sg=_node_surrogate(inst,nodes,cur_uel,cur_ufc,p,seed)
        if sg is None: continue
        Q,c0,k,apply=sg; h,J,const=qubo_to_ising(Q,c0)
        efn=lambda z:qubo_energy(Q,c0,z); ev=energy_vector(efn,k)
        r=optimize_qaoa_x(ev,qp,k,CONFIG['restarts'],seed,CONFIG['maxiter'])
        res['circuit_evals']+=r['nfev']; res['two_qubit_gates']+=2*(k*(k-1)//2)*qp
        res['total_circuit_depth']+=(2*(k-1)+2)*qp
        loc=None
        for z in set(sample_bits(r['probs'],k,shots,rng)):
            tu,tf=apply(z); c=lp_recourse(inst,tu,tf,p); res['recourse_calls']+=1
            if c is not None and (loc is None or c<loc[0]): loc=(c,tu,tf)
        if loc: _,cur_uel,cur_ufc=loc
    pre=lp_recourse(inst,cur_uel,cur_ufc,p); res['recourse_calls']+=1
    if pre is None:
        return dict(status='infeasible',uel=None,ufc=None,cost_pre_repair=None,cost_post_repair=None,resources=res)
    c=pre; bu=list(cur_uel); bf=list(cur_ufc); improved=True
    while improved:
        improved=False
        for i in range(T):
            for arr in (bu,bf):
                arr[i]^=1; c2=lp_recourse(inst,bu,bf,p); res['recourse_calls']+=1
                if c2 is not None and c2<c-1e-9: c=c2; improved=True
                else: arr[i]^=1
    return dict(status='ok',uel=tuple(bu),ufc=tuple(bf),cost_pre_repair=pre,cost_post_repair=c,resources=res)

# ============================ H3: controlled XY vs X (SAME one-hot init) ============================
def mixer_experiment(seed=0,shots=1024):
    """One-hot electrolyser modes {off,low,high}, 3 hours -> 9 qubits.
    BOTH X and XY start from the SAME valid one-hot state (isolates mixer effect)."""
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    Thr=3; n=3*Thr; groups=[[3*t,3*t+1,3*t+2] for t in range(Thr)]
    rng=np.random.default_rng(seed); h=rng.normal(0,0.5,n)
    for g in groups: h[g[1]]-=0.4; h[g[2]]-=0.8
    def onehot_valid(z): return all(sum(z[q] for q in g)==1 for g in groups)
    def run(mixer):
        qc=QuantumCircuit(n)
        for g in groups: qc.x(g[0])            # SAME one-hot init for both
        beta=0.7; gamma=0.5
        for i in range(n):
            if abs(h[i])>1e-12: qc.rz(2*gamma*h[i],i)
        if mixer=='X':
            for i in range(n): qc.rx(2*beta,i)
        else:
            for g in groups:
                for a,b in zip(g,g[1:]+g[:1]): qc.rxx(2*beta,a,b); qc.ryy(2*beta,a,b)
        sv=Statevector.from_instruction(qc)
        probs=(np.abs(sv.data)**2).reshape([2]*n).transpose(list(range(n))[::-1]).reshape(-1)
        rr=np.random.default_rng(1); ks=rr.choice(len(probs),size=shots,p=probs/probs.sum())
        samp=[tuple((int(k)>>(n-1-q))&1 for q in range(n)) for k in ks]
        return float(np.mean([onehot_valid(z) for z in samp]))
    return dict(init='one-hot (both)',X_valid_rate=run('X'),XY_valid_rate=run('XY'),note='controlled for initialization (H3)')


In [5]:
# Cell 3 — FULL authoritative reproduction (all 14 sections) -> results_authoritative.json
import os
WORK='/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.chdir(WORK); os.makedirs('figures',exist_ok=True); os.makedirs('tables',exist_ok=True)
t0=time.time(); SEED=CONFIG['seed']
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
def deps():
    import highspy,qiskit,networkx,scipy,pandas
    return dict(python=sys.version.split()[0],numpy=np.__version__,scipy=scipy.__version__,
        pandas=pandas.__version__,networkx=networkx.__version__,qiskit=qiskit.__version__,
        highspy=getattr(highspy,'__version__','?'))
def ising_energy(h,J,const,z):
    s=1-2*np.asarray(z); e=const
    for i in range(len(s)):
        e+=h[i]*s[i]
        for j in range(i+1,len(s)): e+=J[i,j]*s[i]*s[j]
    return e
def sa_candidates(inst,nn,budget,seed):
    Q,c0,_,_=build_qubo_surrogate(inst,seed=seed); efn=lambda z:qubo_energy(Q,c0,z)
    rng=np.random.default_rng(seed); z=list(rng.integers(0,2,nn)); e=efn(z); cand=set()
    for _ in range(budget):
        i=rng.integers(nn); z2=z[:]; z2[i]^=1; e2=efn(z2)
        if e2<e or rng.random()<np.exp(-(e2-e)/max(abs(e),1)): z=z2; e=e2
        cand.add(tuple(z))
    return cand

csv=find_opsd(); df=build_year(csv,seed=SEED); dev,val,test=split_chrono(df); R={}

inst=make_instance(dev,'2019-01-15 06:00',6)
ex=milp_exact(inst); hi=milp_highs(inst); _,sched=lp_recourse(inst,ex['uel'],ex['ufc'],return_sched=True)
da=demand_audit(inst,ex['uel'],ex['ufc']); br=balance_residuals(inst,sched)
R['E0']=dict(opt_6h=round(ex['J'],4),feasible_patterns=ex['feasible_patterns'],total_patterns=ex['total_patterns'],
    resid_elec=br['elec'],resid_heat=br['heat'],resid_h2=br['h2'],dual_formulation_agree=bool(abs(ex['J']-hi['J'])<1e-6),
    pct_h2_served=da['pct_H2_served'],pct_heat_served=da['pct_heat_served'],
    simultaneous_grid_hrs=da['simultaneous_grid_hrs'],simultaneous_el_fc_hrs=da['simultaneous_el_fc_hrs']); print('[1/10] E0')

R['data']=dict(rows=len(df),neg_price_hours=int((df['price_eur_kwh']<0).sum()),dev=len(dev),val=len(val),test=len(test))

neg_days=sorted(set(df[df['price_eur_kwh']<0].index.floor('D'))); stat={}
for d in neg_days[:20]:
    st=milp_highs(make_instance(df,str(d),24))['status']; stat[st]=stat.get(st,0)+1
R['H1']=dict(days_checked=min(20,len(neg_days)),status_breakdown=stat,unbounded=stat.get('Unbounded',0)); print('[2/10] H1')

Q,c0,n,fit=build_qubo_surrogate(inst,seed=SEED); h,J,const=qubo_to_ising(Q,c0)
qerr=max(abs(qubo_energy(Q,c0,z)-ising_energy(h,J,const,z)) for z in itertools.product([0,1],repeat=n))
sp=[]; rm=[]
for day,spl in [('2019-01-15 06:00',dev),('2019-05-20 06:00',dev),('2019-09-12 06:00',val)]:
    _,_,_,f=build_qubo_surrogate(make_instance(spl,day,6),seed=SEED); sp.append(f['spearman_oos']); rm.append(f['oos_rmse'])
R['encoding']=dict(qubo_ising_maxerr=qerr,spearman_min=round(min(sp),3),spearman_max=round(max(sp),3),
    oos_rmse_min=round(min(rm),2),oos_rmse_max=round(max(rm),2))
allz=list(itertools.product([0,1],repeat=12)); true=[lp_recourse(inst,z[:6],z[6:]) for z in allz]
fe=[(z,c) for z,c in zip(allz,true) if c is not None]; pred=[qubo_energy(Q,c0,z) for z,_ in fe]; tc=[c for _,c in fe]
ot=np.argsort(tc); op=np.argsort(pred)
R['surrogate']=dict(top1_recall=int(len(set(ot[:1])&set(op[:1]))),top5_recall=round(len(set(ot[:5])&set(op[:5]))/5,2),
    regret_eur=round(float(tc[op[0]]-tc[ot[0]]),3)); print('[3/10] encoding+surrogate')

mx=mixer_experiment(seed=SEED)
R['mixer']=dict(X_valid_rate=round(mx['X_valid_rate'],4),XY_valid_rate=round(mx['XY_valid_rate'],4),init='one-hot both (controlled)'); print('[4/10] mixer')

R['partition']={}
for T in [6,12,24,48]:
    G=build_interaction_graph(make_instance(dev,'2019-01-15 00:00',T))
    R['partition'][f'T{T}']=dict(h2aware_cut=partition_metrics(G,get_partition('h2aware',G,T,6))['cut_ratio'],
        temporal_cut=partition_metrics(G,get_partition('temporal',G,T,6))['cut_ratio'],R_Q=round(100*(1-min(12,2*T)/(2*T)),1))
print('[5/10] partition')

base24=make_instance(dev,'2019-01-15 00:00',24)
def bind(mult,H0,Hterm,Hmax):
    i=dict(base24); i['h2d']=base24['h2d']*mult; p=dict(PARAMS); p['H0']=H0; p['Hterm']=Hterm; p['Hmax']=Hmax
    r=milp_highs(i,p); _,s=lp_recourse(i,r['uel'],r['ufc'],p,return_sched=True); Hi=lambda t:24*9+t
    Hs=np.array([s[Hi(t)] for t in range(25)]); return r['J'],int((Hs<=p['Hmin']+1e-6).sum()),i
Jb,loact,ib=bind(5.0,2,24,28); Jbase,_,_=bind(1.0,10,10,50); Gb=build_interaction_graph(ib); imps=[]
for day in ['2019-01-15 00:00','2019-02-10 00:00','2019-03-05 00:00']:
    b=make_instance(dev,day,24); i=dict(b); i['h2d']=b['h2d']*5.0; G=build_interaction_graph(i)
    t=partition_metrics(G,get_partition('temporal',G,24,6))['cut_ratio']; hh=partition_metrics(G,get_partition('h2aware',G,24,6))['cut_ratio']; imps.append(round(100*(t-hh)/t,1))
R['binding']=dict(base_cost=round(Jbase,1),hard_cost=round(Jb,1),lower_bound_active_hrs=loact,
    h2aware_cut=partition_metrics(Gb,get_partition('h2aware',Gb,24,6))['cut_ratio'],
    temporal_cut=partition_metrics(Gb,get_partition('temporal',Gb,24,6))['cut_ratio'],cut_improvement_pct=imps); print('[6/10] binding')

ref24=milp_highs(base24)['J']; G24=build_interaction_graph(base24); R['H14']={'milp_24h':round(ref24,2)}
for nm in ['temporal','h2aware','random','carrier','metis-like']:
    hy=hybrid_solve(base24,get_partition(nm,G24,24,6),qp=2,shots=64,seed=1)
    R['H14'][nm]=dict(gap=round(100*(hy['cost_post_repair']-ref24)/abs(ref24),3),recourse=hy['resources']['recourse_calls'],
        peak_qubits=hy['resources']['peak_qubits'],two_qubit_gates=hy['resources']['two_qubit_gates'],circuit_evals=hy['resources']['circuit_evals'])
print('[7/10] H14')

T=6; LB=ex['J']; gap=lambda c:100*(c-LB)/abs(LB); ev=energy_vector(lambda z:qubo_energy(Q,c0,z),n)
qa=[]; sa=[]
for sd in range(20):
    r=optimize_qaoa_x(ev,3,n,4,sd,200); qs=set(sample_bits(r['probs'],n,256,np.random.default_rng(100+sd)))
    qf=[lp_recourse(inst,z[:T],z[T:]) for z in qs]; qa.append(gap(min([c for c in qf if c is not None])))
    sc=sa_candidates(inst,n,256,sd); sf=[lp_recourse(inst,z[:T],z[T:]) for z in sc]; sa.append(gap(min([c for c in sf if c is not None])))
R['stats_20seed']=dict(instance='2019-01-15 06:00',QAOA_median=round(float(np.median(qa)),3),SA_median=round(float(np.median(sa)),3),wilcoxon_p=round(float(wilcoxon(qa,sa).pvalue),4),n=20)
qg=[]; sg=[]
for day in ['2019-01-15 06:00','2019-03-10 06:00','2019-05-20 06:00','2019-07-15 06:00','2019-08-25 06:00']:
    ins=make_instance(dev,day,6); lb=milp_exact(ins)['J']; g=lambda c:100*(c-lb)/abs(lb)
    Qi,ci,ni,_=build_qubo_surrogate(ins,seed=SEED); evi=energy_vector(lambda z:qubo_energy(Qi,ci,z),ni)
    r=optimize_qaoa_x(evi,3,ni,4,1,200); qs=set(sample_bits(r['probs'],ni,256,np.random.default_rng(1)))
    qg.append(g(min([c for c in (lp_recourse(ins,z[:6],z[6:]) for z in qs) if c is not None])))
    sc=sa_candidates(ins,ni,256,1); sg.append(g(min([c for c in (lp_recourse(ins,z[:6],z[6:]) for z in sc) if c is not None])))
R['stats_5instance']=dict(QAOA_median=round(float(np.median(qg)),3),SA_median=round(float(np.median(sg)),3),wilcoxon_p=round(float(wilcoxon(qg,sg).pvalue),4),n=5); print('[8/10] statistics')

qs=list(set(sample_bits(optimize_qaoa_x(ev,3,n,4,1,200)['probs'],n,512,np.random.default_rng(1))))
qf=[lp_recourse(inst,z[:6],z[6:]) for z in qs]; qff=[c for c in qf if c is not None]
R['E4']=dict(LB_6h=round(LB,4),QAOA_feas_rate=round(len(qff)/len(qs),3),QAOA_gap=round(gap(min(qff)),3))
held=[]
for day in ['2019-11-10 06:00','2019-12-05 06:00','2019-12-20 06:00']:
    ins=make_instance(test,day,6); exi=milp_exact(ins); G=build_interaction_graph(ins)
    hy=hybrid_solve(ins,get_partition('h2aware',G,6,6),qp=3,shots=256,seed=1)
    held.append(round(100*(hy['cost_post_repair']-exi['J'])/abs(exi['J']),3))
R['heldout']=dict(gaps=held,max_gap=max(held)); print('[9/10] E4+heldout')

G48=build_interaction_graph(make_instance(dev,'2019-01-15 00:00',48))
hy48=hybrid_solve(make_instance(dev,'2019-01-15 00:00',48),get_partition('temporal',G48,48,6),qp=2,shots=32,seed=1)
R['scalability']=dict(T48_R_Q=round(100*(1-12/96),1),T48_peak_qubits=hy48['resources']['peak_qubits'],
    T48_two_qubit_gates=hy48['resources']['two_qubit_gates'],T48_circuit_evals=hy48['resources']['circuit_evals']); print('[10/10] scalability')

R['_meta']=dict(model='kaggle_hagdcp.py',seed=SEED,timestamp=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    runtime_s=round(time.time()-t0,1),dataset_file=csv,dataset_sha256=sha256_file(csv),dependencies=deps(),
    params={k:PARAMS[k] for k in PARAMS},sections=[k for k in R if k!='_meta'])
content=json.dumps({k:v for k,v in R.items() if k!='_meta'},sort_keys=True,default=float).encode()
R['_meta']['results_sha256']=hashlib.sha256(content).hexdigest()
json.dump(R,open('results_authoritative.json','w'),indent=2,default=float)
print('\nWROTE results_authoritative.json (%d sections, %.1fs)'%(len(R['_meta']['sections']),R['_meta']['runtime_s']))
print('sections:',R['_meta']['sections'])


[1/10] E0
[2/10] H1
[3/10] encoding+surrogate
[4/10] mixer
[5/10] partition
[6/10] binding
[7/10] H14
[8/10] statistics
[9/10] E4+heldout
[10/10] scalability

WROTE results_authoritative.json (14 sections, 185.8s)
sections: ['E0', 'data', 'H1', 'encoding', 'surrogate', 'mixer', 'partition', 'binding', 'H14', 'stats_20seed', 'stats_5instance', 'E4', 'heldout', 'scalability']


In [6]:
# Cell 4 — generate ALL figures (PNG, 300 DPI) from data + authoritative results
import json, numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':9,'savefig.dpi':300,'savefig.bbox':'tight','axes.grid':True,'grid.alpha':0.3})
Rj=json.load(open('results_authoritative.json'))
B,Q,O='#2c6fbb','#2ca02c','#e8821a'
def sv(fig,name): fig.savefig(f'figures/{name}.png'); plt.close(fig)

# Fig1 electricity week
wk=df.loc['2019-01-14':'2019-01-20']
fig,ax=plt.subplots(figsize=(7,3)); ax.plot(wk.index,wk['load_kw'],label='Load',color=B)
ax.plot(wk.index,wk['solar_kw'],label='Solar',color=O); ax.plot(wk.index,wk['wind_kw'],label='Wind',color=Q)
ax.set_ylabel('kW'); ax.set_title('Electricity demand & renewables (week)'); ax.legend(ncol=3,fontsize=8); fig.autofmt_xdate(); sv(fig,'fig01_electricity_week')
# Fig2 price
fig,ax=plt.subplots(1,2,figsize=(8,3)); ax[0].hist(df['price_eur_kwh']*1000,bins=60,color=B); ax[0].axvline(0,color='k',ls='--',lw=0.8)
ax[0].set_xlabel('EUR/MWh'); ax[0].set_ylabel('hours'); ax[0].set_title('Price distribution')
mp=df['price_eur_kwh'].groupby(df.index.month).mean()*1000; ax[1].bar(mp.index,mp.values,color=B); ax[1].set_xlabel('month'); ax[1].set_title('Monthly mean price'); sv(fig,'fig02_price')
# Fig3 heat+COP
fig,ax=plt.subplots(figsize=(7,3)); ax.plot(wk.index,wk['heat_kwth'],color='#d62728',label='Heat (synthetic)')
ax2=ax.twinx(); ax2.plot(wk.index,wk['cop'],color='#8c564b',label='COP'); ax.set_ylabel('kWth'); ax2.set_ylabel('COP')
ax.set_title('Thermal demand & COP (synthetic)'); fig.autofmt_xdate(); sv(fig,'fig03_heat_cop')
# Fig4 H2 demand
fig,ax=plt.subplots(figsize=(7,2.8)); ax.plot(wk.index,wk['h2_demand_kg_h'],color='#9467bd'); ax.set_ylabel('kg/h'); ax.set_title('Hydrogen demand (simulated)'); fig.autofmt_xdate(); sv(fig,'fig04_h2_demand')
# Fig5 correlation
cols=['load_kw','solar_kw','wind_kw','price_eur_kwh','heat_kwth','cop','h2_demand_kg_h']; cm=df[cols].corr().values
fig,ax=plt.subplots(figsize=(5,4.2)); im=ax.imshow(cm,cmap='RdBu_r',vmin=-1,vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols,rotation=45,ha='right',fontsize=7); ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols,fontsize=7)
for i in range(len(cols)):
    for j in range(len(cols)): ax.text(j,i,f'{cm[i,j]:.2f}',ha='center',va='center',fontsize=6)
fig.colorbar(im,fraction=0.046); ax.set_title('Feature correlation'); sv(fig,'fig05_correlation')
# Fig6 qubit reduction vs horizon
Ts=[6,12,24,48]; full=[2*t for t in Ts]; peak=[min(12,2*t) for t in Ts]
fig,ax=plt.subplots(figsize=(5.5,3.4)); ax.plot(Ts,full,'o-',label='monolithic 2T',color=O); ax.plot(Ts,peak,'s-',label='decomposed peak',color=Q)
for t,f,p in zip(Ts,full,peak): ax.annotate(f'{100*(1-p/f):.0f}%',(t,p),textcoords='offset points',xytext=(0,-12),fontsize=8,color=Q)
ax.set_xlabel('horizon T (h)'); ax.set_ylabel('peak logical qubits'); ax.set_title('Structural qubit reduction'); ax.legend(fontsize=8); sv(fig,'fig06_qubit_reduction')
# Fig7 mixer
fig,ax=plt.subplots(figsize=(4,3.2)); r=Rj['mixer']; ax.bar(['X-mixer','XY-mixer'],[r['X_valid_rate']*100,r['XY_valid_rate']*100],color=[O,Q])
for i,v in enumerate([r['X_valid_rate']*100,r['XY_valid_rate']*100]): ax.text(i,v+2,f'{v:.1f}%',ha='center')
ax.set_ylabel('valid one-hot (%)'); ax.set_title('Mixer (controlled init)'); sv(fig,'fig07_mixer')
# Fig8 statistics QAOA vs SA
s20,s5=Rj['stats_20seed'],Rj['stats_5instance']
fig,ax=plt.subplots(figsize=(6,3.4)); x=np.arange(2); w=0.35
ax.bar(x-w/2,[s20['QAOA_median'],s5['QAOA_median']],w,label='QAOA',color=Q)
ax.bar(x+w/2,[s20['SA_median'],s5['SA_median']],w,label='SA',color=O)
ax.set_xticks(x); ax.set_xticklabels([f'20-seed\np={s20["wilcoxon_p"]}',f'5-instance\np={s5["wilcoxon_p"]}'])
ax.set_ylabel('median gap (%)'); ax.set_title('QAOA vs SA (no significant advantage)'); ax.legend(fontsize=8); sv(fig,'fig08_stats')
# Fig9 H14 end-to-end
parts=['temporal','h2aware','random','carrier','metis-like']; gaps=[Rj['H14'][p]['gap'] for p in parts]; rec=[Rj['H14'][p]['recourse'] for p in parts]
fig,ax=plt.subplots(1,2,figsize=(9,3.2)); ax[0].bar(parts,gaps,color=Q); ax[0].set_ylabel('post-repair gap (%)'); ax[0].set_title('H14: all partitions reach optimum'); ax[0].tick_params(axis='x',rotation=30)
ax[1].bar(parts,rec,color=B); ax[1].set_ylabel('recourse LP calls'); ax[1].set_title('effort per partition'); ax[1].tick_params(axis='x',rotation=30); fig.tight_layout(); sv(fig,'fig09_h14_endtoend')
# Fig10 binding inventory
base=make_instance(dev,'2019-01-15 00:00',24); Hi=lambda t:24*9+t
fig,ax=plt.subplots(figsize=(7,3.2))
for scen,(mult,H0,Ht,Hm,col) in {'base':(1,10,10,50,B),'binding':(5,2,24,28,Q)}.items():
    i=dict(base); i['h2d']=base['h2d']*mult; p=dict(PARAMS); p['H0']=H0; p['Hterm']=Ht; p['Hmax']=Hm
    r=milp_highs(i,p); _,s=lp_recourse(i,r['uel'],r['ufc'],p,return_sched=True)
    ax.plot(range(25),[s[Hi(t)] for t in range(25)],'o-',ms=3,color=col,label=f'{scen} (Hmax={Hm})'); ax.axhline(Hm,color=col,ls=':',lw=0.8)
ax.set_xlabel('hour'); ax.set_ylabel('H2 inventory (kg)'); ax.legend(fontsize=8); ax.set_title('Hydrogen inventory: base vs binding'); sv(fig,'fig10_binding_inventory')
# Fig11 24h dispatch
m=milp_highs(base); _,s=lp_recourse(base,m['uel'],m['ufc'],return_sched=True); nv=9; idx=lambda t,j:t*nv+j
fig,ax=plt.subplots(2,1,figsize=(7,5),sharex=True)
ax[0].stackplot(range(24),[s[idx(t,2)] for t in range(24)],[s[idx(t,3)] for t in range(24)],[s[idx(t,0)] for t in range(24)],[s[idx(t,5)] for t in range(24)],labels=['solar','wind','grid+','fuel cell'],alpha=0.8)
ax[0].plot(range(24),base['load'],'k--',label='load'); ax[0].set_ylabel('kW'); ax[0].legend(ncol=5,fontsize=7); ax[0].set_title('Electricity dispatch (24h optimum)')
ax[1].plot(range(25),[s[Hi(t)] for t in range(25)],'o-',color='#9467bd'); ax[1].set_ylabel('H2 (kg)'); ax[1].set_xlabel('hour'); ax[1].set_title('Hydrogen inventory'); fig.tight_layout(); sv(fig,'fig11_dispatch')
print('figures written:', len(os.listdir('figures')))

figures written: 11


In [7]:
# Cell 5 — generate ALL tables as CSV from the authoritative results
import csv, json
Rj=json.load(open('results_authoritative.json'))
def wcsv(name,header,rows):
    with open(f'tables/{name}.csv','w',newline='') as f:
        w=csv.writer(f); w.writerow(header); w.writerows(rows)
E=Rj['E0']
wcsv('T1_dataset',['field','value'],[['source','OPSD time_series 2020-10-06'],['rows',Rj['data']['rows']],
    ['negative_price_hours',Rj['data']['neg_price_hours']],['dev/val/test',f"{Rj['data']['dev']}/{Rj['data']['val']}/{Rj['data']['test']}"]])
wcsv('T2_milp_validation',['metric','value'],[['opt_6h_EUR',E['opt_6h']],['feasible_patterns',f"{E['feasible_patterns']}/{E['total_patterns']}"],
    ['resid_elec',E['resid_elec']],['resid_heat',E['resid_heat']],['resid_h2',E['resid_h2']],['dual_formulation_agree',E['dual_formulation_agree']],
    ['pct_H2_served',E['pct_h2_served']],['pct_heat_served',E['pct_heat_served']],['simultaneous_import_export_hrs',E['simultaneous_grid_hrs']]])
wcsv('T3_partition_qubit',['T','h2aware_cut','temporal_cut','R_Q_%'],[[T,Rj['partition'][f'T{T}']['h2aware_cut'],Rj['partition'][f'T{T}']['temporal_cut'],Rj['partition'][f'T{T}']['R_Q']] for T in [6,12,24,48]])
wcsv('T4_encoding',['metric','value'],[['qubo_ising_maxerr',Rj['encoding']['qubo_ising_maxerr']],['spearman_min',Rj['encoding']['spearman_min']],
    ['spearman_max',Rj['encoding']['spearman_max']],['oos_rmse_min',Rj['encoding']['oos_rmse_min']],['oos_rmse_max',Rj['encoding']['oos_rmse_max']],
    ['top5_recall',Rj['surrogate']['top5_recall']],['regret_eur',Rj['surrogate']['regret_eur']]])
wcsv('T5_mixer',['mixer','valid_onehot_rate'],[['X',Rj['mixer']['X_valid_rate']],['XY',Rj['mixer']['XY_valid_rate']]])
wcsv('T6_H14_endtoend',['partition','post_repair_gap_%','recourse_calls','peak_qubits','two_qubit_gates'],
    [[p,Rj['H14'][p]['gap'],Rj['H14'][p]['recourse'],Rj['H14'][p]['peak_qubits'],Rj['H14'][p]['two_qubit_gates']] for p in ['temporal','h2aware','random','carrier','metis-like']])
wcsv('T7_binding',['metric','value'],[['base_cost',Rj['binding']['base_cost']],['hard_cost',Rj['binding']['hard_cost']],
    ['lower_bound_active_hrs',Rj['binding']['lower_bound_active_hrs']],['h2aware_cut',Rj['binding']['h2aware_cut']],['temporal_cut',Rj['binding']['temporal_cut']],
    ['cut_improvement_pct',';'.join(map(str,Rj['binding']['cut_improvement_pct']))]])
wcsv('T8_statistics',['comparison','QAOA_median','SA_median','wilcoxon_p','n'],
    [['20-seed',Rj['stats_20seed']['QAOA_median'],Rj['stats_20seed']['SA_median'],Rj['stats_20seed']['wilcoxon_p'],Rj['stats_20seed']['n']],
     ['5-instance',Rj['stats_5instance']['QAOA_median'],Rj['stats_5instance']['SA_median'],Rj['stats_5instance']['wilcoxon_p'],Rj['stats_5instance']['n']]])
wcsv('T9_scalability',['metric','value'],[['T48_R_Q_%',Rj['scalability']['T48_R_Q']],['T48_peak_qubits',Rj['scalability']['T48_peak_qubits']],
    ['T48_two_qubit_gates',Rj['scalability']['T48_two_qubit_gates']],['T48_circuit_evals',Rj['scalability']['T48_circuit_evals']],['heldout_max_gap_%',Rj['heldout']['max_gap']]])
print('tables written:', len(os.listdir('tables')), '->', sorted(os.listdir('tables')))

tables written: 9 -> ['T1_dataset.csv', 'T2_milp_validation.csv', 'T3_partition_qubit.csv', 'T4_encoding.csv', 'T5_mixer.csv', 'T6_H14_endtoend.csv', 'T7_binding.csv', 'T8_statistics.csv', 'T9_scalability.csv']


In [8]:
# Cell 6 — regression tests (edge cases)
def t_neg():
    nd=sorted(set(df[df['price_eur_kwh']<0].index.floor('D')))[0]; assert milp_highs(make_instance(df,str(nd),24))['status']=='Optimal'
def t_loss():
    p=dict(PARAMS); p['self_discharge']=0.02; assert milp_highs(make_instance(dev,'2019-01-15 00:00',24),p)['J'] is not None
def t_demand():
    i=make_instance(dev,'2019-01-15 06:00',6); e=milp_exact(i); d=demand_audit(i,e['uel'],e['ufc']); assert d['pct_H2_served']>99.99
def t_cache():
    i=make_instance(dev,'2019-01-15 06:00',6); j=dict(i); j['price']=i['price'][::-1].copy(); assert _inst_hash(i,PARAMS)!=_inst_hash(j,PARAMS)
for t in [t_neg,t_loss,t_demand,t_cache]: t(); print('PASS',t.__name__)

PASS t_neg
PASS t_loss
PASS t_demand
PASS t_cache


In [20]:

# ================================================================
# HA-GDCP-QAOA — COMPLETE EVIDENCE GENERATOR (SINGLE KAGGLE CELL)
# ================================================================

import os, sys, json, re, hashlib, zipfile, shutil
import subprocess, time, platform, importlib.metadata, traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
from scipy.stats import wilcoxon

DATASET = Path(
    "/kaggle/input/datasets/sunawarkhan/reproducibility-package"
)
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
AUDIT = WORK / "HA_GDCP_QAOA_EVIDENCE"
SOURCE = AUDIT / "original_sources"
RUN = AUDIT / "fresh_run"

for folder in (WORK, AUDIT, SOURCE, RUN):
    folder.mkdir(parents=True, exist_ok=True)

EXPECTED_OPSD_SHA = (
    "6a7f2bc571314cbf9c321cc03437691cd4be95c3a6f075e60ff99e8035c704c8"
)

checks = []

def report(name, status, detail=""):
    entry = {
        "check": name,
        "status": status,
        "detail": str(detail)
    }
    checks.append(entry)
    print(f"[{status}] {name}: {detail}")

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def results_checksum(data):
    # Original reproduce_all.py hashes ONLY non-_meta sections
    payload = {
        k: v for k, v in data.items()
        if k != "_meta"
    }
    calculated = hashlib.sha256(
        json.dumps(
            payload, sort_keys=True, default=float
        ).encode()
    ).hexdigest()
    stored = data.get("_meta", {}).get("results_sha256")
    return stored == calculated, stored, calculated

def find_any(root, names):
    for name in names:
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    return None

def safe_extract(zp, dest):
    with zipfile.ZipFile(zp) as z:
        for item in z.infolist():
            target = (dest / item.filename).resolve()
            if not target.is_relative_to(dest.resolve()):
                raise ValueError(
                    f"Unsafe ZIP member: {item.filename}"
                )
        z.extractall(dest)

print("=" * 72)
print("HA-GDCP-QAOA REPRODUCIBILITY EVIDENCE GENERATOR")
print("=" * 72)

# ------------------------------------------------
# STEP 1: Discover attached source dataset
# ------------------------------------------------

roots = []

if DATASET.exists():
    roots.append(DATASET)
    report("Source dataset", "PASS", str(DATASET))
else:
    report(
        "Source dataset",
        "WARNING",
        "Exact path missing; searching other Kaggle inputs"
    )

if INPUT.exists():
    roots.append(INPUT)

roots.append(WORK)

# Extract ZIPs from the uploaded reproducibility dataset
extracted = AUDIT / "uploaded_package"
extracted.mkdir(exist_ok=True)

zips = sorted(DATASET.rglob("*.zip")) if DATASET.exists() else []

for zp in zips:
    try:
        safe_extract(zp, extracted)
        report("Package extraction", "PASS", zp.name)
    except Exception as exc:
        report("Package extraction", "FAIL", repr(exc))

roots.insert(0, extracted)

# ------------------------------------------------
# STEP 2: Locate original source files
# ------------------------------------------------

required = {
    "kaggle_hagdcp.py": [
        "kaggle_hagdcp.py",
        "kaggle_hagdcp(1).py"
    ],
    "reproduce_all.py": [
        "reproduce_all.py",
        "reproduce_all(1).py",
        "reproduce_all(2).py"
    ],
    "check_claims.py": [
        "check_claims.py",
        "check_claims(1).py",
        "check_claims(2).py"
    ],
    "manuscript.md": [
        "manuscript.md",
        "manuscript(1).md"
    ]
}

located = {}

for standard, variants in required.items():
    found = None

    for root in roots:
        found = find_any(root, variants)
        if found:
            break

    if found:
        destination = SOURCE / standard
        if found.resolve() != destination.resolve():
            shutil.copy2(found, destination)
        located[standard] = destination

        report(
            "Source: " + standard,
            "PASS",
            f"SHA256={file_sha(destination)}"
        )
    else:
        located[standard] = None
        report(
            "Source: " + standard,
            "NOT VERIFIED",
            "File not found"
        )

# ------------------------------------------------
# STEP 3: Locate OPSD CSV and previous results
# ------------------------------------------------

csv = None
for root in roots:
    csv = find_any(
        root, ["time_series_60min_singleindex.csv"]
    )
    if csv:
        break

# Search ZIP archives for OPSD CSV without extracting
# unrelated files
if csv is None and INPUT.exists():
    for zp in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(zp) as z:
                candidates = [
                    n for n in z.namelist()
                    if n.endswith("time_series_60min_singleindex.csv")
                ]
                if candidates:
                    csv = RUN / "time_series_60min_singleindex.csv"
                    with z.open(candidates[0]) as src, open(csv, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    break
        except (zipfile.BadZipFile, OSError):
            continue

if csv:
    digest = file_sha(csv)
    report(
        "OPSD original data SHA256",
        "PASS" if digest == EXPECTED_OPSD_SHA else "FAIL",
        f"path={csv}; sha256={digest}"
    )
else:
    report(
        "OPSD original data SHA256",
        "NOT VERIFIED",
        "OPSD CSV missing. Attach the original OPSD dataset."
    )

previous_path = None

for root in roots:
    previous_path = find_any(
        root,
        [
            "results_authoritative.json",
            "results_authoritative(2).json",
            "results_authoritative(1).json"
        ]
    )
    if previous_path:
        break

OLD = None

if previous_path:
    try:
        OLD = json.loads(previous_path.read_text())
        shutil.copy2(
            previous_path,
            AUDIT / "previous_results.json"
        )
        ok, stored, calculated = results_checksum(OLD)
        report(
            "Previous JSON checksum",
            "PASS" if ok else "FAIL",
            f"stored={stored}; calculated={calculated}"
        )
    except Exception as exc:
        report("Previous JSON", "FAIL", repr(exc))
else:
    report(
        "Previous JSON",
        "NOT VERIFIED",
        "No previous authoritative results attached"
    )

# ------------------------------------------------
# STEP 4: Capture environment
# ------------------------------------------------

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

for package in [
    "numpy", "scipy", "pandas", "networkx",
    "highspy", "qiskit", "qiskit-aer"
]:
    try:
        environment[package] = importlib.metadata.version(
            package
        )
    except importlib.metadata.PackageNotFoundError:
        environment[package] = "NOT INSTALLED"

(AUDIT / "environment.json").write_text(
    json.dumps(environment, indent=2)
)

# ------------------------------------------------
# STEP 5: Prepare fresh independent run
# ------------------------------------------------

runner_src = located["reproduce_all.py"]
model_src = located["kaggle_hagdcp.py"]

can_run = bool(runner_src and model_src and csv)

if can_run:
    shutil.copy2(model_src, RUN / "kaggle_hagdcp.py")
    shutil.copy2(runner_src, RUN / "reproduce_all_original.py")

    script = runner_src.read_text(encoding="utf-8")

    # Add raw arrays to statistical results BEFORE checksum.
    # Original source remains unchanged.
    markers = {
        "R['stats_20seed']=dict(": (
            "R['stats_20seed']=dict("
        ),
        "R['stats_5instance']=dict(": (
            "R['stats_5instance']=dict("
        )
    }

    marker20 = "R['stats_20seed']=dict("
    marker5 = "R['stats_5instance']=dict("
    meta_marker = "R['_meta']=dict("

    if all(
        script.count(m) == 1
        for m in [marker20, marker5, meta_marker]
    ):
        # Save raw arrays without changing existing summary math
        script = script.replace(
            marker20,
            """
RAW20 = dict(
    seeds=list(range(20)),
    qaoa_gaps_pct=[float(x) for x in qa],
    sa_gaps_pct=[float(x) for x in sa]
)
""" + marker20,
            1
        )

        script = script.replace(
            marker5,
            """
RAW5 = dict(
    instances=[
        "2019-01-15 06:00",
        "2019-03-10 06:00",
        "2019-05-20 06:00",
        "2019-07-15 06:00",
        "2019-08-25 06:00"
    ],
    qaoa_gaps_pct=[float(x) for x in qg],
    sa_gaps_pct=[float(x) for x in sg]
)
""" + marker5,
            1
        )

        script = script.replace(
            meta_marker,
            """
R['stats_20seed']['raw'] = RAW20
R['stats_5instance']['raw'] = RAW5

""" + meta_marker,
            1
        )

        try:
            compile(script, "reproduce_all.py", "exec")
            (RUN / "reproduce_all.py").write_text(
                script, encoding="utf-8"
            )
            report(
                "Raw statistics instrumentation",
                "PASS",
                "20 seeds and 5 instances captured"
            )
        except SyntaxError as exc:
            can_run = False
            report(
                "Raw statistics instrumentation",
                "FAIL",
                repr(exc)
            )
    else:
        can_run = False
        report(
            "Raw statistics instrumentation",
            "FAIL",
            "Original script structure differs from expected"
        )

# ------------------------------------------------
# STEP 6: Execute all experiments
# ------------------------------------------------

fresh_path = RUN / "results_authoritative.json"

if can_run:
    env = os.environ.copy()
    env["PYTHONHASHSEED"] = "0"
    env["OPSD_CSV"] = str(csv.resolve())
    env["PYTHONPATH"] = str(RUN.resolve())

    print("\nRUNNING FRESH EXPERIMENTS...")
    print("Please wait for completion.\n")

    start = time.time()

    with open(
        AUDIT / "reproduction.log",
        "w",
        encoding="utf-8"
    ) as log:
        proc = subprocess.run(
            [sys.executable, "reproduce_all.py"],
            cwd=str(RUN),
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True
        )

    runtime = round(time.time() - start, 2)

    report(
        "Fresh experiment execution",
        "PASS" if proc.returncode == 0 else "FAIL",
        f"exit={proc.returncode}; runtime={runtime}s"
    )

    if proc.returncode != 0:
        lines = (
            AUDIT / "reproduction.log"
        ).read_text(errors="replace").splitlines()

        print("\nERROR LOG:")
        print("\n".join(lines[-35:]))

else:
    report(
        "Fresh experiment execution",
        "NOT VERIFIED",
        "Required source, dataset or instrumentation unavailable"
    )

# ------------------------------------------------
# STEP 7: Verify JSON and raw statistical tests
# ------------------------------------------------

NEW = None

if fresh_path.exists():
    try:
        NEW = json.loads(fresh_path.read_text())

        ok, stored, calculated = results_checksum(NEW)

        report(
            "Fresh authoritative checksum",
            "PASS" if ok else "FAIL",
            f"stored={stored}; calculated={calculated}"
        )

        for section in [
            "stats_20seed",
            "stats_5instance"
        ]:
            try:
                stats = NEW[section]
                raw = stats["raw"]

                qa = np.asarray(
                    raw["qaoa_gaps_pct"], dtype=float
                )
                sa = np.asarray(
                    raw["sa_gaps_pct"], dtype=float
                )

                assert len(qa) == len(sa) == stats["n"]
                assert np.isfinite(qa).all()
                assert np.isfinite(sa).all()

                qmedian = float(np.median(qa))
                smedian = float(np.median(sa))
                pvalue = float(wilcoxon(qa, sa).pvalue)

                valid = (
                    abs(qmedian - stats["QAOA_median"]) <= 0.00051
                    and abs(smedian - stats["SA_median"]) <= 0.00051
                    and abs(pvalue - stats["wilcoxon_p"]) <= 0.000051
                )

                report(
                    section + " raw statistical verification",
                    "PASS" if valid else "FAIL",
                    f"n={len(qa)}; "
                    f"QAOA median={qmedian:.8f}; "
                    f"SA median={smedian:.8f}; "
                    f"Wilcoxon p={pvalue:.8f}"
                )

                (AUDIT / f"{section}_raw.json").write_text(
                    json.dumps(raw, indent=2)
                )

            except Exception as exc:
                report(
                    section + " raw statistical verification",
                    "FAIL",
                    repr(exc)
                )

    except Exception as exc:
        report("Fresh JSON loading", "FAIL", repr(exc))

else:
    report(
        "Fresh authoritative JSON",
        "NOT VERIFIED",
        "No fresh JSON produced"
    )

# ------------------------------------------------
# STEP 8: Manuscript claim verification
# ------------------------------------------------

checker = located["check_claims.py"]
manuscript = located["manuscript.md"]

if NEW is not None and checker and manuscript:
    shutil.copy2(checker, RUN / "check_claims.py")
    shutil.copy2(manuscript, RUN / "manuscript.md")

    # Correct known X-mixer checker typo in audit copy only
    checker_path = RUN / "check_claims.py"
    checker_text = checker_path.read_text()

    checker_text = checker_text.replace(
        "('X_valid_rate (%)',        jpath(R,'mixer.XY_valid_rate')",
        "('X_valid_rate (%)',        jpath(R,'mixer.X_valid_rate')"
    )

    checker_path.write_text(checker_text)

    def execute_checker(extra_args, output_file):
        p = subprocess.run(
            [
                sys.executable,
                "check_claims.py",
                *extra_args
            ],
            cwd=str(RUN),
            capture_output=True,
            text=True
        )

        (AUDIT / output_file).write_text(
            p.stdout + "\n" + p.stderr
        )
        return p

    p = execute_checker(
        ["--manuscript", "manuscript.md"],
        "manuscript_claims.log"
    )

    report(
        "Manuscript numerical claims",
        "PASS" if p.returncode == 0 else "FAIL",
        f"exit={p.returncode}"
    )

    t = execute_checker(
        ["--manuscript", "manuscript.md", "--selftest"],
        "tamper_selftest.log"
    )

    report(
        "Claim checker tamper self-test",
        "PASS" if t.returncode == 0 else "FAIL",
        f"exit={t.returncode}"
    )

else:
    report(
        "Manuscript numerical claims",
        "NOT VERIFIED",
        "Fresh JSON, manuscript or checker unavailable"
    )

# ------------------------------------------------
# STEP 9: Compare previous and fresh results
# ------------------------------------------------

if OLD is not None and NEW is not None:
    sections = [
        ("stats_20seed", "QAOA_median"),
        ("stats_20seed", "SA_median"),
        ("stats_20seed", "wilcoxon_p"),
        ("stats_5instance", "QAOA_median"),
        ("stats_5instance", "SA_median"),
        ("stats_5instance", "wilcoxon_p"),
        ("heldout", "max_gap")
    ]

    comparison = []

    for section, key in sections:
        try:
            before = OLD[section][key]
            after = NEW[section][key]

            equal = abs(float(before) - float(after)) <= 0.000051

            comparison.append({
                "section": section,
                "field": key,
                "previous": before,
                "fresh": after,
                "match": equal
            })

            report(
                f"Previous vs fresh: {section}.{key}",
                "PASS" if equal else "WARNING",
                f"previous={before}; fresh={after}"
            )

        except Exception as exc:
            report(
                f"Previous vs fresh: {section}.{key}",
                "NOT VERIFIED",
                repr(exc)
            )

    (AUDIT / "results_comparison.json").write_text(
        json.dumps(comparison, indent=2)
    )

# ------------------------------------------------
# STEP 10: Export final audit report
# ------------------------------------------------

summary = {
    status: sum(
        x["status"] == status for x in checks
    )
    for status in [
        "PASS", "FAIL", "WARNING", "NOT VERIFIED"
    ]
}

final_report = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset_path": str(DATASET),
    "checks": checks,
    "summary": summary,
    "environment": environment
}

(AUDIT / "verification_report.json").write_text(
    json.dumps(final_report, indent=2)
)

with open(
    AUDIT / "verification_report.txt",
    "w",
    encoding="utf-8"
) as f:
    for item in checks:
        f.write(
            f"[{item['status']}] "
            f"{item['check']}: "
            f"{item['detail']}\n"
        )

    f.write("\nSUMMARY\n")
    f.write(json.dumps(summary, indent=2))

# ------------------------------------------------
# STEP 11: Create final evidence ZIP
# ------------------------------------------------

zip_path = WORK / "HA_GDCP_QAOA_COMPLETE_EVIDENCE.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for p in AUDIT.rglob("*"):
        if p.is_file():
            # Avoid packaging large original OPSD CSV
            if p.name == "time_series_60min_singleindex.csv":
                continue

            z.write(
                p,
                arcname=p.relative_to(AUDIT)
            )

print("\n" + "=" * 72)
print("FINAL AUDIT SUMMARY")
print("=" * 72)
print(json.dumps(summary, indent=2))

if summary["FAIL"]:
    print("VERDICT: FAIL — see logs.")
elif summary["NOT VERIFIED"]:
    print("VERDICT: PARTIAL — missing evidence.")
elif summary["WARNING"]:
    print("VERDICT: REVIEW DIFFERENCES.")
else:
    print("VERDICT: ALL CONFIGURED CHECKS PASSED.")

print("\nEVIDENCE FILE:")
print(zip_path)




HA-GDCP-QAOA REPRODUCIBILITY EVIDENCE GENERATOR
[PASS] Source dataset: /kaggle/input/datasets/sunawarkhan/reproducibility-package
[PASS] Source: kaggle_hagdcp.py: SHA256=f912a1a90f3bdc3243dbcab6a02c21f0e2096f91d5806002fb76b410148a8ef6
[PASS] Source: reproduce_all.py: SHA256=bc618894102b55f1673f9759f2fe0b7e41434852c12669d42ab21149d791f026
[PASS] Source: check_claims.py: SHA256=4dc70cdb59f40aa1e3fb16b1f272863168455644d3305830201467521c506898
[PASS] Source: manuscript.md: SHA256=d4bb317708c1d14a823864b95fdc6e363b9604c8c006eefcdad9d3846a38b185
[PASS] OPSD original data SHA256: path=/kaggle/input/datasets/sunawarkhan/opsd-timeseriessunawar/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv; sha256=6a7f2bc571314cbf9c321cc03437691cd4be95c3a6f075e60ff99e8035c704c8
[FAIL] Previous JSON checksum: stored=8d09f4a02bba6accdc3bc8f53c830b8c4d0cf46502d53faf1111d4f6c54f9065; calculated=262286ad0cf4a6d8510f92f82d50673f0eab4b400bf5a5b346d135cac692b0d1
[PASS] Raw statistics instrumentation: 20 

In [18]:

import json
from pathlib import Path

base = Path("/kaggle/working/HA_GDCP_QAOA_EVIDENCE")

for name in [
    "verification_report.txt",
    "manuscript_claims.log",
    "tamper_selftest.log"
]:
    p = base / name
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(p.read_text(errors="replace") if p.exists() else "NOT FOUND")

log = base / "reproduction.log"
if log.exists():
    print("\nREPRODUCTION LOG — LAST 40 LINES")
    print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))



verification_report.txt
[PASS] Source dataset: /kaggle/input/datasets/sunawarkhan/reproducibility-package
[PASS] Source: kaggle_hagdcp.py: SHA256=f912a1a90f3bdc3243dbcab6a02c21f0e2096f91d5806002fb76b410148a8ef6
[PASS] Source: reproduce_all.py: SHA256=bc618894102b55f1673f9759f2fe0b7e41434852c12669d42ab21149d791f026
[PASS] Source: check_claims.py: SHA256=4dc70cdb59f40aa1e3fb16b1f272863168455644d3305830201467521c506898
[PASS] Source: manuscript.md: SHA256=d4bb317708c1d14a823864b95fdc6e363b9604c8c006eefcdad9d3846a38b185
[PASS] OPSD original data SHA256: path=/kaggle/input/datasets/sunawarkhan/opsd-timeseriessunawar/opsd-time_series-2020-10-06/time_series_60min_singleindex.csv; sha256=6a7f2bc571314cbf9c321cc03437691cd4be95c3a6f075e60ff99e8035c704c8
[FAIL] Previous JSON checksum: stored=8d09f4a02bba6accdc3bc8f53c830b8c4d0cf46502d53faf1111d4f6c54f9065; calculated=262286ad0cf4a6d8510f92f82d50673f0eab4b400bf5a5b346d135cac692b0d1
[PASS] Raw statistics instrumentation: 20 seeds and 5 instances c

In [19]:

import json
from pathlib import Path
from pprint import pprint

p = Path(
    "/kaggle/working/HA_GDCP_QAOA_EVIDENCE/"
    "fresh_run/results_authoritative.json"
)

R = json.loads(p.read_text())

print("HELD-OUT RESULTS")
pprint(R["heldout"], sort_dicts=False)

print("\n20-SEED RESULTS")
pprint({
    k: v for k, v in R["stats_20seed"].items()
    if k != "raw"
}, sort_dicts=False)


HELD-OUT RESULTS
{'gaps': [0.0, 0.0, 0.0], 'max_gap': 0.0}

20-SEED RESULTS
{'instance': '2019-01-15 06:00',
 'QAOA_median': 0.176,
 'SA_median': 0.415,
 'wilcoxon_p': 0.0973,
 'n': 20}


## What you get (3 outputs, all in `hagdcp_outputs.zip`)
1. **`results_authoritative.json`** — complete results (14 sections) with provenance (data + result SHA-256, seed, dependency versions, runtime).
2. **`figures/` (11 PNG, 300 DPI)** — electricity, price, heat/COP, H2, correlation, qubit-reduction, mixer, QAOA-vs-SA, H14 end-to-end, binding inventory, dispatch.
3. **`tables/` (9 CSV)** — dataset, MILP validation, partition/qubit, encoding, mixer, H14, binding, statistics, scalability.

All numbers match the manuscript (seed = 2026). No quantum advantage is claimed; classical repair dominates.

In [21]:

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")

if token:
    print("GitHub Token Connected Successfully!")
else:
    print("Token not found")

GitHub Token Connected Successfully!


In [22]:

from pathlib import Path
import shutil

WORK = Path("/kaggle/working")
PROJECT = WORK / "github_export"
PROJECT.mkdir(exist_ok=True)

# Copy research outputs
for item in WORK.iterdir():
    if item.name in ["github_export", ".virtual_documents"]:
        continue

    target = PROJECT / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    elif item.is_file():
        shutil.copy2(item, target)

print("Research files prepared!")

# Check dataset sizes
for dataset in Path("/kaggle/input").iterdir():
    size = sum(
        p.stat().st_size
        for p in dataset.rglob("*") if p.is_file()
    ) if dataset.is_dir() else dataset.stat().st_size

    print(dataset.name, round(size / 1024**2, 2), "MB")


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Research files prepared!
datasets 467.12 MB


In [25]:

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

print("Token connected!" if token else "Token missing!")


Token connected!


In [27]:

import requests
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

repo = "SunawarKhan/HA-GDCP-QAOA-Research"

response = requests.get(
    f"https://api.github.com/repos/{repo}",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    },
    timeout=30
)

print("HTTP Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Repository:", data["full_name"])
    print("Push Permission:", data["permissions"]["push"])
else:
    print("Access check failed.")



HTTP Status: 200
Repository: SunawarKhan/HA-GDCP-QAOA-Research
Push Permission: True


In [33]:

from pathlib import Path
import shutil

WORK = Path("/kaggle/working")
PROJECT = WORK / "github_export"

PROJECT.mkdir(exist_ok=True)

for item in WORK.iterdir():
    if item.name in ["github_export", ".virtual_documents"]:
        continue

    destination = PROJECT / item.name

    if item.is_dir():
        shutil.copytree(
            item, destination, dirs_exist_ok=True
        )
    elif item.is_file():
        shutil.copy2(item, destination)

print("All research outputs prepared!")

for item in PROJECT.iterdir():
    print(item.name)


All research outputs prepared!
results_authoritative.json
HA_GDCP_QAOA_COMPLETE_EVIDENCE.zip
hagdcp_outputs.zip
verification_report.txt
reproduce_all.py
tables
check_claims.py
HA_GDCP_QAOA_EVIDENCE
.git
manuscript.md
verification_report.json
kaggle_hagdcp.py
figures


In [35]:

import os
import subprocess
import tempfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

USERNAME = "SunawarKhan"
REPO = "HA-GDCP-QAOA-Research"
EMAIL = "sunawarkhan@gmail.com"

PROJECT = Path("/kaggle/working/github_export")
TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()

assert PROJECT.is_dir(), "Run Step 1 first"

def run_git(*args, env=None):
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT,
        env=env,
        check=True,
        capture_output=True,
        text=True
    )

run_git("init")
run_git("checkout", "-B", "main")
run_git("config", "user.name", USERNAME)
run_git("config", "user.email", EMAIL)

run_git("add", ".")
status = run_git("status", "--porcelain").stdout

if status:
    run_git("commit", "-m", "Upload Kaggle research outputs")

REMOTE = f"https://github.com/{USERNAME}/{REPO}.git"

with tempfile.TemporaryDirectory() as tmp:
    askpass = Path(tmp) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        ' *Username*) echo "x-access-token" ;;\n'
        ' *Password*) printf "%s\\n" "$GH_TOKEN" ;;\n'
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GH_TOKEN"] = TOKEN

    try:
        run_git("push", "-u", REMOTE, "main", env=env)
        print("SUCCESS! GitHub upload completed.")
    except subprocess.CalledProcessError:
        print("Push failed. Check token permissions,")
        print("repository history and file sizes.")

Push failed. Check token permissions,
repository history and file sizes.


In [31]:
import json
from pathlib import Path

for p in Path("/kaggle/working").rglob("results_authoritative.json"):
    R = json.loads(p.read_text())
    print("\nFILE:", p)
    print("SHA256:", R.get("_meta", {}).get("results_sha256"))
    print("20-SEED:", R.get("stats_20seed"))
    print("HELD-OUT:", R.get("heldout"))


FILE: /kaggle/working/results_authoritative.json
SHA256: 7cd114f587e0780f5095a9c8b7c802793fe106f00a353bebbcc3d545529c2862
20-SEED: {'instance': '2019-01-15 06:00', 'QAOA_median': 0.176, 'SA_median': 0.415, 'wilcoxon_p': 0.0973, 'n': 20}
HELD-OUT: {'gaps': [0.0, 0.0, 0.0], 'max_gap': 0.0}

FILE: /kaggle/working/github_export/results_authoritative.json
SHA256: 7cd114f587e0780f5095a9c8b7c802793fe106f00a353bebbcc3d545529c2862
20-SEED: {'instance': '2019-01-15 06:00', 'QAOA_median': 0.176, 'SA_median': 0.415, 'wilcoxon_p': 0.0973, 'n': 20}
HELD-OUT: {'gaps': [0.0, 0.0, 0.0], 'max_gap': 0.0}

FILE: /kaggle/working/github_export/HA_GDCP_QAOA_EVIDENCE/fresh_run/results_authoritative.json
SHA256: ffae6743c58a418311fc66b01cd4fba06bf03834a5fd63cb10703bd5cdbdb282
20-SEED: {'instance': '2019-01-15 06:00', 'QAOA_median': 0.176, 'SA_median': 0.415, 'wilcoxon_p': 0.0973, 'n': 20, 'raw': {'seeds': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], 'qaoa_gaps_pct': [0.1815773259120